# Aggregated win rate vs sample count

This notebook reads `result_3_*.json` (aggregate win-rate stats) and plots win rate vs average sample count across LLMs.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

llms = ['gemma2_9b', 'llama3.1_8b', 'mistral_7b', 'qwen2.5_7b']
rms = ['fsfairx_rm', 'mistral_rm']

llm_mapping = {
    'gemma2_9b': 'google/gemma-2-9b-it',
    'llama3.1_8b': 'meta-llama/Llama-3.1-8B-Instruct',
    'llama3.2_3b': 'meta-llama/Llama-3.2-3B-Instruct',
    'mistral_7b': 'mistralai/Mistral-7B-Instruct-v0.3',
    'qwen2.5_7b': 'Qwen/Qwen2.5-7B-Instruct',
    'qwen3_4b': 'Qwen/Qwen3-4B-Instruct-2507',
}

rm_mapping = {
    'fsfairx_rm': 'sfairXC/FsfairX-LLaMA3-RM-v0.1',
    'mistral_rm': 'weqweasdas/RM-Mistral-7B',
}


In [ ]:
# Config
dataset = 'alpaca'
rm = 'fsfairx_rm'
distribution = 'shifted_exponential'
transformation = 'cdf'
batch_size = '1'
alpha = '0.99'

# Pick cost indices from data['costs']
cost_indices = [0, 3, 6, 9]  # adjust as needed


In [ ]:
def _load_aggregate_for_llm(llm_name):
    path = Path(f'../slurm_result_{dataset}/result_3_{rm}_{llm_name}_{distribution}_{transformation}_bs{batch_size}_a{alpha}.json')
    with path.open() as f:
        data = json.load(f)

    costs = data['costs']
    aggregate = data['aggregate']

    points = []
    for i in cost_indices:
        entry = aggregate[i]
        points.append({
            'cost': costs[i],
            'avg_sample_count_median': entry['avg_sample_count_median'],
            'avg_sample_count_p25': entry['avg_sample_count_p25'],
            'avg_sample_count_p75': entry['avg_sample_count_p75'],
            'win_rate_median': entry['win_rate_median'],
            'win_rate_p25': entry['win_rate_p25'],
            'win_rate_p75': entry['win_rate_p75'],
        })

    return points


In [ ]:
def build_points_by_llm():
    points_by_llm = {}
    for llm_name in llms:
        points_by_llm[llm_name] = _load_aggregate_for_llm(llm_name)
    return points_by_llm


In [ ]:
def plot_win_rate_vs_samples(points_by_llm, rm_name, dataset_name):
    fig, ax = plt.subplots(figsize=(8, 6))

    for llm_name, points in points_by_llm.items():
        xs = np.array([p['avg_sample_count_median'] for p in points], dtype=float)
        ys = np.array([p['win_rate_median'] for p in points], dtype=float)
        yerr = np.vstack([
            ys - np.array([p['win_rate_p25'] for p in points], dtype=float),
            np.array([p['win_rate_p75'] for p in points], dtype=float) - ys,
        ])

        order = np.argsort(xs)
        xs = xs[order]
        ys = ys[order]
        yerr = yerr[:, order]

        ax.errorbar(
            xs,
            ys,
            yerr=yerr,
            marker='o',
            linewidth=1.5,
            markersize=5,
            capsize=3,
            label=llm_mapping.get(llm_name, llm_name),
        )

    ax.set_xlabel('Average Sample Count (median)', fontsize=12)
    ax.set_ylabel('Win Rate (median)', fontsize=12)
    dataset_label = 'AlpacaEval' if dataset_name == 'alpaca' else 'HH-RLHF'
    ax.set_title(
        f'Win Rate vs Sample Count | RM: {rm_mapping.get(rm_name, rm_name)} | Dataset: {dataset_label}',
        fontsize=12,
    )
    ax.grid(True, alpha=0.3)
    ax.legend(title='LLM Models')

    fig.tight_layout()
    plt.show()
    return fig


In [ ]:
points_by_llm = build_points_by_llm()
plot_win_rate_vs_samples(points_by_llm, rm_name=rm, dataset_name=dataset)
